In [1]:
import os
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression

In [2]:
candidate_paths = [
    "student_data_cleaned.csv",
    "StudentPerformanceFactors.csv",
    "student_data.csv",
    os.path.join("data", "student_data_cleaned.csv"),
    os.path.join("..", "data", "student_data_cleaned.csv")
]

data_path = None
for p in candidate_paths:
    if os.path.exists(p):
        data_path = p
        break

if data_path is None:
    raise FileNotFoundError("Could not find clean dataset. Ensure 'student_data_cleaned.csv' is available.")

df = pd.read_csv(data_path)

In [3]:
if df['Teacher_Quality'].isnull().sum() > 0:
    df['Teacher_Quality'] = df['Teacher_Quality'].fillna(df['Teacher_Quality'].mode()[0])
if df['Parental_Education_Level'].isnull().sum() > 0:
    df['Parental_Education_Level'] = df['Parental_Education_Level'].fillna(df['Parental_Education_Level'].mode()[0])
if df['Distance_from_Home'].isnull().sum() > 0:
    df['Distance_from_Home'] = df['Distance_from_Home'].fillna(df['Distance_from_Home'].mode()[0])
df.loc[df['Exam_Score'] > 100, 'Exam_Score'] = 100

In [4]:
X = df.drop(columns=['Exam_Score'])
y = df['Exam_Score']

print(f"Dataset successfully loaded: {df.shape[0]} records.")
print(f"Features matrix (X) shape: {X.shape}")
print(f"Target vector (y) shape:    {y.shape}")

Dataset successfully loaded: 6607 records.
Features matrix (X) shape: (6607, 19)
Target vector (y) shape:    (6607,)


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print("--- Train/Test Split Partition Summary ---")
print(f"Total student records: {len(df)}")
print(f"Training set (80%):    {X_train.shape[0]} records")
print(f"Testing set (20%):     {X_test.shape[0]} records")

--- Train/Test Split Partition Summary ---
Total student records: 6607
Training set (80%):    5285 records
Testing set (20%):     1322 records


In [6]:
# Identify column categories
num_features = X.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X.select_dtypes(include=['object']).columns.tolist()

# ColumnTransformer: Scale numeric features, One-Hot Encode categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_features)
    ],
    remainder='drop'
)

# Fit preprocessor strictly on training data
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Train the Baseline Linear Regression Model
baseline_model = LinearRegression()
baseline_model.fit(X_train_processed, y_train)

print("✓ Baseline Linear Regression model trained successfully.")
print(f"  Processed feature count: {X_train_processed.shape[1]}")
print(f"  Model Intercept (bias):  {baseline_model.intercept_:.2f}")

✓ Baseline Linear Regression model trained successfully.
  Processed feature count: 27
  Model Intercept (bias):  68.58


In [7]:
# Generate initial test predictions
y_pred_test = baseline_model.predict(X_test_processed)

# Display sample predictions vs actual values
sample_preview = pd.DataFrame({
    'Actual Exam Score': y_test.values[:5],
    'Predicted Exam Score': np.round(y_pred_test[:5], 2),
    'Absolute Difference': np.round(np.abs(y_test.values[:5] - y_pred_test[:5]), 2)
})

print("--- Sample Predictions Preview (First 5 Test Students) ---")
display(sample_preview)

# Ensure model directory exists and save artifact
model_dir = "model" if os.path.exists("model") else os.path.join("..", "model")
os.makedirs(model_dir, exist_ok=True)

model_file = os.path.join(model_dir, "baseline_linear_model.joblib")
joblib.dump(baseline_model, model_file)

if os.path.exists(model_file):
    print(f"\n✓ Saved baseline model artifact to: {model_file} ({os.path.getsize(model_file)/1024:.1f} KB)")

--- Sample Predictions Preview (First 5 Test Students) ---


,Actual Exam Score,Predicted Exam Score,Absolute Difference
0,65,64.53,0.47
1,65,65.26,0.26
2,71,71.53,0.53
3,64,64.28,0.28
4,66,66.52,0.52



✓ Saved baseline model artifact to: ../model/baseline_linear_model.joblib (1.0 KB)
